# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haritharamadass/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)



## 1. My rule and its reason codes


In [1]:
# ============================================================
# ML-07 — Section 1
# Load March/April data and check two baseline signals.
#
# March 2026 = information available at decision time
# April 2026 = future outcome used ONLY for evaluation
# ============================================================

from google.colab import userdata
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Connect to the FlyRank warehouse
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found. Add it to Colab Secrets first."
    )

print("HF token loaded successfully:", HF_TOKEN is not None)


%pip -q install duckdb

import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB connected to Hugging Face successfully.")


# ------------------------------------------------------------
# 2. Define March feature window and April outcome window
# ------------------------------------------------------------

MARCH = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

APRIL = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""


# ------------------------------------------------------------
# 3. Build one row per client/content item
#
# Monthly search position is calculated as an
# impression-weighted average of daily gsc_avg_position.
#
# Only March information becomes a candidate baseline input.
# April is used only to create the future evaluation outcome.
# ------------------------------------------------------------

feature_frame = con.sql(f"""
WITH march AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_31d,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_31d,

        SUM(
            gsc_avg_position * gsc_impressions
        ) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS avg_position_31d,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS active_gsc_days

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_impressions,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days

    FROM {APRIL}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.impressions_31d,
    m.clicks_31d,

    100.0 * m.clicks_31d
        / NULLIF(m.impressions_31d, 0) AS ctr_31d,

    m.avg_position_31d,

    m.active_gsc_days,

    (
        a.april_impressions
        < 0.80 * m.impressions_31d
    ) AS declined_next_month

FROM march m

JOIN april a
    USING (
        client_hash_id,
        content_hash_id
    )

WHERE
    m.impressions_31d >= 100
    AND m.active_gsc_days > 0
    AND a.april_gsc_days > 0
    AND m.impressions_31d IS NOT NULL
    AND m.avg_position_31d IS NOT NULL
    AND m.avg_position_31d > 0
    AND a.april_impressions IS NOT NULL
""").df()


# ------------------------------------------------------------
# 4. Basic verification
# ------------------------------------------------------------

print("\nFeature frame created successfully.")
print("Rows available for ML-07:", len(feature_frame))

print(
    "Average-position range:",
    round(feature_frame["avg_position_31d"].min(), 3),
    "to",
    round(feature_frame["avg_position_31d"].max(), 3)
)

print("\nColumns:")
print(feature_frame.columns.tolist())


# ------------------------------------------------------------
# Keep positive position values as observed in the warehouse.
# We only reject zero or negative values.
# ------------------------------------------------------------

positions_at_or_below_zero = (
    feature_frame["avg_position_31d"] <= 0
).sum()

positions_below_one = (
    (feature_frame["avg_position_31d"] > 0)
    & (feature_frame["avg_position_31d"] < 1)
).sum()

print(
    "Rows with average position <= 0:",
    positions_at_or_below_zero
)

print(
    "Positive rows with average position below 1:",
    positions_below_one
)

assert positions_at_or_below_zero == 0, (
    "Unexpected zero or negative search position."
)


# ------------------------------------------------------------
# Check missing values
# ------------------------------------------------------------

print("\nMissing values in key fields:")

print(
    feature_frame[
        [
            "impressions_31d",
            "clicks_31d",
            "ctr_31d",
            "avg_position_31d",
            "active_gsc_days",
            "declined_next_month"
        ]
    ].isna().sum()
)


# ============================================================
# SIGNAL CHECK 1 — CTR vs search position
#
# This is linked to FlyRank's CTR-vs-position logic.
# ============================================================

signal1 = feature_frame.copy()


signal1["position_bucket"] = pd.cut(
    signal1["avg_position_31d"],
    bins=[
        0,
        3,
        10,
        20,
        50,
        np.inf
    ],
    labels=[
        "<=3",
        "4-10",
        "11-20",
        "21-50",
        "51+"
    ],
    include_lowest=True
)


signal1_table = (
    signal1
    .dropna(subset=["position_bucket"])
    .groupby(
        "position_bucket",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr_31d", "median"),
        mean_ctr=("ctr_31d", "mean"),
        decline_rate=("declined_next_month", "mean")
    )
    .reset_index()
)


signal1_table["median_ctr"] = (
    signal1_table["median_ctr"].round(3)
)

signal1_table["mean_ctr"] = (
    signal1_table["mean_ctr"].round(3)
)

signal1_table["decline_rate"] = (
    100 * signal1_table["decline_rate"]
).round(2)


print("\n" + "=" * 70)
print("SIGNAL 1 — CTR vs SEARCH POSITION")
print("=" * 70)

print(
    "Total n =",
    int(signal1_table["n"].sum())
)

display(signal1_table)


# ============================================================
# SIGNAL CHECK 2 — Search impression volume
#
# Tests whether volume itself is associated with the future
# decline outcome and whether it is useful mainly for
# prioritization / quick-win decisions.
# ============================================================

signal2 = feature_frame.copy()


signal2["impression_bucket"] = pd.cut(
    signal2["impressions_31d"],
    bins=[
        99,
        249,
        499,
        999,
        np.inf
    ],
    labels=[
        "100-249",
        "250-499",
        "500-999",
        "1000+"
    ],
    include_lowest=True
)


signal2_table = (
    signal2
    .groupby(
        "impression_bucket",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_31d", "median"),
        median_ctr=("ctr_31d", "median"),
        decline_rate=("declined_next_month", "mean")
    )
    .reset_index()
)


signal2_table["median_impressions"] = (
    signal2_table["median_impressions"].round(0)
)

signal2_table["median_ctr"] = (
    signal2_table["median_ctr"].round(3)
)

signal2_table["decline_rate"] = (
    100 * signal2_table["decline_rate"]
).round(2)


print("\n" + "=" * 70)
print("SIGNAL 2 — SEARCH IMPRESSION VOLUME")
print("=" * 70)

print(
    "Total n =",
    int(signal2_table["n"].sum())
)

display(signal2_table)


# ------------------------------------------------------------
# 5. Leakage statement
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

print(
    "March fields are candidate baseline inputs."
)

print(
    "declined_next_month is created from April and is used "
    "only for retrospective evaluation, never to calculate "
    "the baseline score, reason code, or action."
)

print(
    "No future-window field or label-derived product flag "
    "is used as a baseline feature."
)

HF token loaded successfully: True
DuckDB connected to Hugging Face successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature frame created successfully.
Rows available for ML-07: 100893
Average-position range: 0.02 to 115.662

Columns:
['client_hash_id', 'content_hash_id', 'impressions_31d', 'clicks_31d', 'ctr_31d', 'avg_position_31d', 'active_gsc_days', 'declined_next_month']
Rows with average position <= 0: 0
Positive rows with average position below 1: 573

Missing values in key fields:
impressions_31d        0
clicks_31d             0
ctr_31d                0
avg_position_31d       0
active_gsc_days        0
declined_next_month    0
dtype: int64

SIGNAL 1 — CTR vs SEARCH POSITION
Total n = 100893


,position_bucket,n,median_ctr,mean_ctr,decline_rate
0,<=3,9732,0.219,0.347,55.39
1,4-10,47445,0.192,0.321,50.90
2,11-20,19678,0.100,0.244,52.11
3,21-50,19913,0.000,0.142,49.81
4,51+,4125,0.000,0.046,54.11



SIGNAL 2 — SEARCH IMPRESSION VOLUME
Total n = 100893


,impression_bucket,n,median_impressions,median_ctr,decline_rate
0,100-249,21901,160.0,0.000,53.48
1,250-499,17146,353.0,0.000,54.32
2,500-999,16820,701.0,0.143,53.65
3,1000+,45026,2944.0,0.194,48.63



LEAKAGE CHECK
March fields are candidate baseline inputs.
declined_next_month is created from April and is used only for retrospective evaluation, never to calculate the baseline score, reason code, or action.
No future-window field or label-derived product flag is used as a baseline feature.


### Signal checks and baseline rule

**Signal 1 — CTR vs search position: CONFIRMED**

In this March slice, CTR generally decreases as average search position becomes worse. Median CTR is 0.219 in the top-position bucket (≤3), falls to 0.192 for positions 4–10 and 0.100 for positions 11–20, and reaches 0.000 in the lower-position buckets. This supports using CTR relative to search position as a directional opportunity signal. It does not prove that position causes CTR differences.

**Signal 2 — search impression volume: MIXED**

Impression volume does not show a consistent increase in the future decline rate. The 1000+ impression bucket has a 48.63% decline rate, compared with roughly 53–54% in the smaller-volume buckets. Therefore, volume alone is not a reliable decline signal. I will use it as a prioritization signal because higher-volume content can represent a larger opportunity when another weakness is also present.

### My baseline rule

I will prioritize content that already has meaningful search visibility but appears to have a CTR opportunity relative to its search position. Higher impression volume increases priority, while CTR and position determine whether the item looks actionable.

The rule uses only March information available at the decision point. April's `declined_next_month` outcome is used only to evaluate the ranked baseline and is never used to calculate the score.

### Reason codes

- `LOW_CTR_HIGH_VISIBILITY` — meaningful impressions with CTR weaker than expected for the item's position.
- `HIGH_VOLUME_OPPORTUNITY` — strong search visibility combined with an actionable CTR/position opportunity.
- `LOW_PRIORITY` — the page does not meet the main opportunity conditions.

### Action labels

- `REVIEW_CTR` — review the page for title, snippet, or content improvements.
- `MONITOR` — keep the page in the queue but do not prioritize it now.

## 2. Build the ranked queue (writes the CSV)


In [2]:
# ============================================================
# ML-07 — Section 2
# Build a transparent baseline score and ranked action queue
# ============================================================

import os
import numpy as np
import pandas as pd

baseline = feature_frame.copy()


# ------------------------------------------------------------
# 1. CTR opportunity relative to current search position
# These are simple, hand-written thresholds.
# ------------------------------------------------------------

baseline["ctr_opportunity"] = (
    (
        (baseline["avg_position_31d"] <= 3)
        & (baseline["ctr_31d"] < 0.15)
    )
    |
    (
        (baseline["avg_position_31d"] > 3)
        & (baseline["avg_position_31d"] <= 10)
        & (baseline["ctr_31d"] < 0.10)
    )
    |
    (
        (baseline["avg_position_31d"] > 10)
        & (baseline["avg_position_31d"] <= 20)
        & (baseline["ctr_31d"] < 0.05)
    )
)


# ------------------------------------------------------------
# 2. Transparent score
#
# +3 = CTR opportunity
# +2 = 1000+ impressions
# +1 = 500-999 impressions
# +1 = currently ranking in top 20
# ------------------------------------------------------------

baseline["score"] = (
    3 * baseline["ctr_opportunity"].astype(int)
    + 2 * (baseline["impressions_31d"] >= 1000).astype(int)
    + 1 * (
        (baseline["impressions_31d"] >= 500)
        & (baseline["impressions_31d"] < 1000)
    ).astype(int)
    + 1 * (baseline["avg_position_31d"] <= 20).astype(int)
)


# ------------------------------------------------------------
# 3. Exactly ONE reason code per row
# ------------------------------------------------------------

baseline["reason_code"] = np.select(
    [
        baseline["ctr_opportunity"]
        & (baseline["impressions_31d"] >= 1000),

        baseline["ctr_opportunity"]
        & (baseline["impressions_31d"] >= 500),
    ],
    [
        "HIGH_VOLUME_OPPORTUNITY",
        "LOW_CTR_HIGH_VISIBILITY",
    ],
    default="LOW_PRIORITY"
)


# ------------------------------------------------------------
# 4. Action label
# ------------------------------------------------------------

baseline["action"] = np.where(
    baseline["ctr_opportunity"]
    & (baseline["impressions_31d"] >= 500),
    "REVIEW_CTR",
    "MONITOR"
)


# ------------------------------------------------------------
# 5. Rank the complete queue
# Higher score first, then higher impressions
# ------------------------------------------------------------

baseline = (
    baseline
    .sort_values(
        by=["score", "impressions_31d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

baseline["rank"] = np.arange(1, len(baseline) + 1)


# ------------------------------------------------------------
# 6. Write the required CSV
# ------------------------------------------------------------

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

output_columns = [
    "rank",
    "content_hash_id",
    "impressions_31d",
    "clicks_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
    "score",
    "reason_code",
    "action"
]

baseline[output_columns].to_csv(
    output_path,
    index=False
)

print("CSV written successfully:")
print(output_path)

print("\nRows ranked:", len(baseline))


# ------------------------------------------------------------
# 7. Honest evaluation
# April outcome is used here ONLY after scoring.
# ------------------------------------------------------------

base_rate = baseline["declined_next_month"].mean()

top20 = baseline.head(20)

precision_at_20 = (
    top20["declined_next_month"]
    .astype(int)
    .mean()
)

print("\nEvaluation")
print("-" * 50)
print(f"Base decline rate: {base_rate:.3f}")
print(f"Precision@20:      {precision_at_20:.3f}")


# ------------------------------------------------------------
# 8. Preview the ranked queue
# ------------------------------------------------------------

print("\nTop 20 ranked items:")

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "impressions_31d",
            "ctr_31d",
            "avg_position_31d",
            "score",
            "reason_code",
            "action",
            "declined_next_month"
        ]
    ]
)

CSV written successfully:
work/outputs/baseline_action_score.csv

Rows ranked: 100893

Evaluation
--------------------------------------------------
Base decline rate: 0.515
Precision@20:      0.600

Top 20 ranked items:


,rank,content_hash_id,impressions_31d,ctr_31d,avg_position_31d,score,reason_code,action,declined_next_month
0,1,content_44f34c0a90047651,212404.0,0.011299,0.665877,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
1,2,content_8d7d99f109e19aa2,203497.0,0.142017,2.468557,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
2,3,content_34a70fea29d15f24,143019.0,0.030066,3.166132,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
3,4,content_8e1334d6356668e3,134984.0,0.000741,2.693038,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
4,5,content_7c6373141eae744a,132593.0,0.062598,5.948459,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
5,6,content_fec55986a1868d62,124075.0,0.000806,0.308426,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
6,7,content_f6116743b00afc2d,107584.0,0.013943,9.735658,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
7,8,content_cd3d932d4e1c8db0,89332.0,0.004478,7.831807,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
8,9,content_9ef3d7516483e665,89229.0,0.103105,2.410965,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True
9,10,content_9c057b66c30a3abb,83834.0,0.001193,0.116006,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,True


### Ranked queue result

The baseline ranked 100,893 eligible content items. The observed base decline rate is 51.5%, while Precision@20 is 60.0%. This is an 8.5 percentage-point improvement over the base rate at the top of the queue.

I treat this as a modest directional baseline rather than a strong predictive result. Its purpose is to provide a transparent benchmark that the Week-5 model must beat.

## 3. Top-20 review


In [3]:
# ============================================================
# ML-07 — Section 3
# Manual-style Top-20 review
# ============================================================

review = baseline.head(20).copy()


# ------------------------------------------------------------
# Expected CTR threshold used by my baseline
# ------------------------------------------------------------

def ctr_threshold(position):
    if position <= 3:
        return 0.15
    elif position <= 10:
        return 0.10
    elif position <= 20:
        return 0.05
    return np.nan


review["ctr_threshold"] = review["avg_position_31d"].apply(
    ctr_threshold
)

review["ctr_gap"] = (
    review["ctr_threshold"] - review["ctr_31d"]
)


# ------------------------------------------------------------
# Confidence note
# Confidence depends only on March information.
# ------------------------------------------------------------

def confidence_note(row):
    threshold = row["ctr_threshold"]

    if pd.isna(threshold):
        return "Low — outside the main position range."

    # CTR is far below the hand-written threshold
    if row["ctr_31d"] <= 0.50 * threshold:
        return (
            "High — very high visibility and CTR is well below "
            "the rule threshold."
        )

    return (
        "Medium — visibility is high, but CTR is relatively "
        "close to the rule threshold."
    )


review["confidence_note"] = review.apply(
    confidence_note,
    axis=1
)


# ------------------------------------------------------------
# Why each item is in the queue
# ------------------------------------------------------------

review["why_it_is_here"] = review.apply(
    lambda r:
        f"{int(r['impressions_31d']):,} March impressions; "
        f"CTR {r['ctr_31d']:.3f}; "
        f"average position {r['avg_position_31d']:.2f}.",
    axis=1
)


# ------------------------------------------------------------
# What could make each recommendation wrong?
# ------------------------------------------------------------

def what_would_make_it_wrong(row):

    threshold = row["ctr_threshold"]

    if pd.notna(threshold) and row["ctr_31d"] > 0.80 * threshold:
        return (
            "The CTR is only slightly below the hand-written "
            "threshold, so normal variation or query mix could "
            "make this a false priority."
        )

    if row["avg_position_31d"] <= 3:
        return (
            "SERP features, branded or navigational intent, or "
            "query mix could explain the low CTR even if the page "
            "itself does not need a CTR-focused change."
        )

    if row["avg_position_31d"] <= 10:
        return (
            "The current ranking position or query intent could "
            "be the main reason for low CTR, so changing the "
            "title, snippet, or content might not improve clicks."
        )

    return (
        "The page may need a ranking or relevance fix rather "
        "than a CTR-focused review."
    )


review["what_would_make_it_wrong"] = review.apply(
    what_would_make_it_wrong,
    axis=1
)


# ------------------------------------------------------------
# Final Top-20 review table
# ------------------------------------------------------------

top20_review = review[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "why_it_is_here",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()


print("TOP-20 BASELINE REVIEW")
print("=" * 70)
print("Reviewed rows:", len(top20_review))

display(top20_review)


# ------------------------------------------------------------
# One line per ranked item, matching the assignment wording
# ------------------------------------------------------------

print("\nONE-LINE REVIEW FOR EACH TOP-20 ITEM")
print("=" * 70)

for _, row in top20_review.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"Action={row['action']} | "
        f"Reason={row['reason_code']} | "
        f"{row['why_it_is_here']} | "
        f"{row['confidence_note']} | "
        f"What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

TOP-20 BASELINE REVIEW
Reviewed rows: 20


,rank,content_hash_id,action,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_44f34c0a90047651,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"212,404 March impressions; CTR 0.011; average ...",High — very high visibility and CTR is well be...,"SERP features, branded or navigational intent,..."
1,2,content_8d7d99f109e19aa2,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"203,497 March impressions; CTR 0.142; average ...","Medium — visibility is high, but CTR is relati...",The CTR is only slightly below the hand-writte...
2,3,content_34a70fea29d15f24,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"143,019 March impressions; CTR 0.030; average ...",High — very high visibility and CTR is well be...,The current ranking position or query intent c...
3,4,content_8e1334d6356668e3,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"134,984 March impressions; CTR 0.001; average ...",High — very high visibility and CTR is well be...,"SERP features, branded or navigational intent,..."
4,5,content_7c6373141eae744a,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"132,593 March impressions; CTR 0.063; average ...","Medium — visibility is high, but CTR is relati...",The current ranking position or query intent c...
5,6,content_fec55986a1868d62,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"124,075 March impressions; CTR 0.001; average ...",High — very high visibility and CTR is well be...,"SERP features, branded or navigational intent,..."
6,7,content_f6116743b00afc2d,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"107,584 March impressions; CTR 0.014; average ...",High — very high visibility and CTR is well be...,The current ranking position or query intent c...
7,8,content_cd3d932d4e1c8db0,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"89,332 March impressions; CTR 0.004; average p...",High — very high visibility and CTR is well be...,The current ranking position or query intent c...
8,9,content_9ef3d7516483e665,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"89,229 March impressions; CTR 0.103; average p...","Medium — visibility is high, but CTR is relati...","SERP features, branded or navigational intent,..."
9,10,content_9c057b66c30a3abb,REVIEW_CTR,HIGH_VOLUME_OPPORTUNITY,"83,834 March impressions; CTR 0.001; average p...",High — very high visibility and CTR is well be...,"SERP features, branded or navigational intent,..."



ONE-LINE REVIEW FOR EACH TOP-20 ITEM
Rank 1: Action=REVIEW_CTR | Reason=HIGH_VOLUME_OPPORTUNITY | 212,404 March impressions; CTR 0.011; average position 0.67. | High — very high visibility and CTR is well below the rule threshold. | What would make it wrong: SERP features, branded or navigational intent, or query mix could explain the low CTR even if the page itself does not need a CTR-focused change.
Rank 2: Action=REVIEW_CTR | Reason=HIGH_VOLUME_OPPORTUNITY | 203,497 March impressions; CTR 0.142; average position 2.47. | Medium — visibility is high, but CTR is relatively close to the rule threshold. | What would make it wrong: The CTR is only slightly below the hand-written threshold, so normal variation or query mix could make this a false priority.
Rank 3: Action=REVIEW_CTR | Reason=HIGH_VOLUME_OPPORTUNITY | 143,019 March impressions; CTR 0.030; average position 3.17. | High — very high visibility and CTR is well below the rule threshold. | What would make it wrong: The current ra

## 4. Weak picks + leakage check



In [4]:
# ============================================================
# ML-07 — Section 4
# Weak picks + leakage check
# ============================================================

# ------------------------------------------------------------
# 1. Find weak picks among the Top 20
#
# The April outcome is used here only for retrospective
# evaluation. It was NOT used to create the baseline score.
# ------------------------------------------------------------

weak_picks = review[
    review["declined_next_month"] == False
].copy()

weak_picks["threshold_ratio"] = (
    weak_picks["ctr_31d"]
    / weak_picks["ctr_threshold"]
)

print("WEAK PICKS IN TOP 20")
print("=" * 70)

print(
    f"Weak picks (future decline=False): "
    f"{len(weak_picks)} of 20"
)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "impressions_31d",
            "ctr_31d",
            "avg_position_31d",
            "score",
            "reason_code",
            "action",
            "declined_next_month"
        ]
    ]
)


# ------------------------------------------------------------
# 2. Inspect the most questionable weak picks
#
# A row closer to its CTR threshold is more borderline.
# ------------------------------------------------------------

borderline_weak = (
    weak_picks
    .sort_values(
        "threshold_ratio",
        ascending=False
    )
    .head(5)
)

print("\nMOST BORDERLINE WEAK PICKS")
print("=" * 70)

display(
    borderline_weak[
        [
            "rank",
            "impressions_31d",
            "ctr_31d",
            "ctr_threshold",
            "avg_position_31d",
            "threshold_ratio"
        ]
    ]
)


# ------------------------------------------------------------
# 3. Explain why weak picks can happen
# ------------------------------------------------------------

print("\nWeak-pick interpretation:")
print(
    "These rows satisfied the March rule but did not show the "
    "future decline outcome in April. Possible explanations include "
    "query intent, SERP features, branded/navigation traffic, normal "
    "month-to-month variation, or thresholds that are too broad."
)

print(
    "This does not invalidate the baseline; it shows where the "
    "hand-written rule is imperfect and gives the Week-5 model "
    "something concrete to improve."
)


# ------------------------------------------------------------
# 4. Leakage check
# ------------------------------------------------------------

baseline_input_features = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d"
]

future_or_label_fields = [
    "declined_next_month",
    "april_impressions",
    "april_gsc_days"
]

product_flag_fields = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

print("\nLEAKAGE CHECK")
print("=" * 70)

print("Fields used by the baseline rule:")
for field in baseline_input_features:
    print(" -", field)


future_leakage = [
    field
    for field in future_or_label_fields
    if field in baseline_input_features
]

flag_leakage = [
    field
    for field in product_flag_fields
    if field in baseline_input_features
]

print("\nFuture/label-derived inputs used:", future_leakage)
print("Product/label flags used:", flag_leakage)


assert len(future_leakage) == 0, \
    "Future-window leakage detected."

assert len(flag_leakage) == 0, \
    "Product/label flag leakage detected."


print("\nPASS — no future-window or label-derived inputs were used.")
print(
    "April declined_next_month is used only after ranking "
    "for evaluation of the frozen March baseline."
)

WEAK PICKS IN TOP 20
Weak picks (future decline=False): 8 of 20


,rank,content_hash_id,impressions_31d,ctr_31d,avg_position_31d,score,reason_code,action,declined_next_month
1,2,content_8d7d99f109e19aa2,203497.0,0.142017,2.468557,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
3,4,content_8e1334d6356668e3,134984.0,0.000741,2.693038,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
11,12,content_9540d884af3e41fd,82376.0,0.013353,8.005184,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
13,14,content_33d31496fca9665e,79484.0,0.089326,5.334948,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
14,15,content_895d440b9d28c9e1,74063.0,0.095864,5.616124,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
16,17,content_e9f2d0579387d3c3,73503.0,0.089792,5.612193,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
18,19,content_425715547c6a3ea8,71513.0,0.004195,6.983444,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False
19,20,content_c46df0fa61530d86,70398.0,0.059661,0.969332,6,HIGH_VOLUME_OPPORTUNITY,REVIEW_CTR,False



MOST BORDERLINE WEAK PICKS


,rank,impressions_31d,ctr_31d,ctr_threshold,avg_position_31d,threshold_ratio
14,15,74063.0,0.095864,0.10,5.616124,0.958643
1,2,203497.0,0.142017,0.15,2.468557,0.946779
16,17,73503.0,0.089792,0.10,5.612193,0.897923
13,14,79484.0,0.089326,0.10,5.334948,0.893262
19,20,70398.0,0.059661,0.15,0.969332,0.397739



Weak-pick interpretation:
These rows satisfied the March rule but did not show the future decline outcome in April. Possible explanations include query intent, SERP features, branded/navigation traffic, normal month-to-month variation, or thresholds that are too broad.
This does not invalidate the baseline; it shows where the hand-written rule is imperfect and gives the Week-5 model something concrete to improve.

LEAKAGE CHECK
Fields used by the baseline rule:
 - impressions_31d
 - ctr_31d
 - avg_position_31d

Future/label-derived inputs used: []
Product/label flags used: []

PASS — no future-window or label-derived inputs were used.
April declined_next_month is used only after ranking for evaluation of the frozen March baseline.
